# 03 Retries, Timeouts, and Circuit Breakers (LiteLLM, 2026)

## What This Lesson Is
Implement bounded retry logic and a basic circuit breaker for unstable model calls.

## Scientific Lens
- Concept: Transient failure recovery with bounded risk
- Measure: Mean attempts to success and breaker-open frequency
- Validity Limit: Short notebook runs cannot model long-term breaker tuning under real traffic.


## How It Works
1. Simulate transient faults and observe bounded retries.
2. Open breaker after repeated failures.
3. Call live model with strict timeout and retry budget.


In [ ]:
import os
print("OPENAI_API_KEY configured:", bool(os.getenv("OPENAI_API_KEY")))
print("Retry budget:", 3)
print("Timeout seconds:", 8)


## Code Walkthrough
The next two code cells are intentionally split:
- `Deterministic Demo`: always runnable and concept-focused.
- `Live Demo`: executes a real provider/CLI flow with explicit graceful-skip behavior.


In [ ]:
# Deterministic Demo
state = {"attempt": 0, "breaker_open": False}


def flaky_provider():
    state["attempt"] += 1
    if state["attempt"] <= 2:
        raise TimeoutError("simulated timeout")
    return "ok"

max_retries = 3
response = None
for i in range(1, max_retries + 1):
    try:
        response = flaky_provider()
        break
    except TimeoutError as exc:
        print(f"attempt {i} failed: {exc}")
        if i == max_retries:
            state["breaker_open"] = True

print("response:", response, "breaker_open:", state["breaker_open"])
assert response == "ok"


In [ ]:
# Live Demo
import os
import time

try:
    from litellm import completion
except Exception as exc:
    print(f"Skipping live retry demo: litellm unavailable ({exc})")
else:
    api_key = os.getenv("OPENAI_API_KEY")
    if not api_key:
        print("Skipping live retry demo: OPENAI_API_KEY not set.")
    else:
        prompt = "Define circuit breaker pattern in one concise sentence."
        for attempt in range(1, 4):
            try:
                r = completion(
                    model="openai/gpt-4.1-mini",
                    messages=[{"role": "user", "content": prompt}],
                    api_key=api_key,
                    timeout=8,
                )
                print("success attempt:", attempt)
                print(r.choices[0].message.content.strip())
                break
            except Exception as exc:
                print(f"attempt {attempt} failed: {exc}")
                time.sleep(0.2 * attempt)
        else:
            print("Circuit breaker would open here after retry budget exhaustion.")


## Applied Labs
1. Decrease timeout to force more retry failures and observe breaker behavior.
2. Add jitter to backoff and compare retry burst patterns.
3. Track retry attempt histogram across 20 simulated calls.

## Validation Checklist
- Retry loop is bounded (no unbounded while-true retry).
- Timeout is explicitly set on live calls.
- Breaker-open state is observable when retries are exhausted.

## Further Reading
- [LiteLLM Reliability Patterns](https://docs.litellm.ai/docs/routing)
- [Azure Circuit Breaker Pattern](https://learn.microsoft.com/azure/architecture/patterns/circuit-breaker)
- [AWS Exponential Backoff and Jitter](https://aws.amazon.com/builders-library/timeouts-retries-and-backoff-with-jitter/)
